# 05 — Ensemble Models + SMOTE Analysis

**Models:** XGB_weighted, RF_tuned, AdaBoost, XGB_SMOTE  
**Labels:** loaded from `data/processed/train_labels.csv` / `test_labels.csv` (generated by 03c).  
**Key fixes:** Per-label binary relevance, val-set threshold tuning, no shared feature matrix.  
**Key finding:** XGB_weighted (native class-weighting) outperforms SMOTE — reported as a project finding.

In [1]:
import os, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.base import clone
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
    hamming_loss, accuracy_score,
    precision_recall_curve
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

PROCESSED = '../data/processed'
MODELS    = '../models'
PLOTS     = '../results/plots'
METRICS   = '../results/metrics'
for d in [MODELS, PLOTS, METRICS]:
    os.makedirs(d, exist_ok=True)

LABELS = ['Anaemia','Diabetes_Risk','Dyslipidemia','Kidney_Risk','Liver_Stress','Thyroid_Abnormal']
print('Ready.')

Ready.


## Step 1 — Load Data

In [2]:
# FIX B: file-check before loading labels
train_label_path = f'{PROCESSED}/train_labels.csv'
test_label_path  = f'{PROCESSED}/test_labels.csv'

if not os.path.exists(train_label_path) or not os.path.exists(test_label_path):
    raise FileNotFoundError(
        'Run 03c_ClusterRangeLearning.ipynb first to generate '
        'train_labels.csv and test_labels.csv.'
    )

train_scaled    = pd.read_csv(f'{PROCESSED}/train_wide_scaled.csv')
test_scaled     = pd.read_csv(f'{PROCESSED}/test_wide_scaled.csv')
train_clust     = pd.read_csv(f'{PROCESSED}/train_clusters.csv')
test_clust      = pd.read_csv(f'{PROCESSED}/test_clusters.csv')
train_labels_df = pd.read_csv(train_label_path).set_index('document_id')
test_labels_df  = pd.read_csv(test_label_path).set_index('document_id')

train_scaled = train_scaled.merge(train_clust, on='document_id', how='left')
test_scaled  = test_scaled.merge(test_clust,   on='document_id', how='left')

print(f'Train: {train_scaled.shape}  |  Test: {test_scaled.shape}')
print(f'Clusters in train: {sorted(train_scaled["cluster_id"].unique())}')

# FIX D: val split for threshold tuning
train_doc_ids, val_doc_ids = train_test_split(
    train_scaled['document_id'], test_size=0.15, random_state=42
)
train_fit = train_scaled[train_scaled['document_id'].isin(train_doc_ids)].reset_index(drop=True)
train_val = train_scaled[train_scaled['document_id'].isin(val_doc_ids)].reset_index(drop=True)
print(f'Fit: {len(train_fit):,}  |  Val: {len(train_val):,}')

Train: (79993, 110)  |  Test: (19999, 110)
Clusters in train: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


Fit: 67,994  |  Val: 11,999


In [3]:
# FIX C: LABEL_EXCLUDE + feature_cols_for_label helper
LABEL_EXCLUDE = {
    'Anaemia':          ['haemoglobin','hemoglobin','hgb','mcv','mch','mchc','rbc','rdw',
                         'pcv','hematocrit','reticulocyte'],
    'Diabetes_Risk':    ['hba1c','hemoglobin a1c','glycated','glucose','fasting glucose',
                         'pp glucose','blood sugar'],
    'Dyslipidemia':     ['total cholesterol','cholesterol','ldl','hdl','triglyceride',
                         'triglycerides','vldl','lipoprotein'],
    'Kidney_Risk':      ['creatinine','bun','blood urea nitrogen','urea','egfr','gfr','uric acid'],
    'Liver_Stress':     ['alt','sgpt','ast','sgot','ggt','bilirubin','albumin','alp',
                         'alkaline phosphatase','alk phos'],
    'Thyroid_Abnormal': ['tsh','t3','t4','ft3','ft4','thyroid','triiodothyronine','thyroxine'],
}

# IMPROVEMENT: Verbatim copy of feature_cols_for_label from nb04
def feature_cols_for_label(df, label):
    base_exclude = {'document_id', 'age', 'gender'}
    blocked_terms = LABEL_EXCLUDE[label]
    cols = []
    for c in df.columns:
        c_lower = c.lower()
        if c in base_exclude:
            continue
        if any(term.lower() in c_lower for term in blocked_terms):
            continue
        cols.append(c)
    return cols

def all_feature_cols(df):
    base_exclude = {'document_id', 'age', 'gender'}
    return [c for c in df.columns if c not in base_exclude]

# Label arrays aligned by document_id
def get_y(df_scaled, lbl_df):
    return lbl_df[LABELS].reindex(df_scaled['document_id']).values

y_fit  = get_y(train_fit,   train_labels_df)
y_val  = get_y(train_val,   train_labels_df)
y_test = get_y(test_scaled, test_labels_df)
print(f'y_fit: {y_fit.shape}  y_val: {y_val.shape}  y_test: {y_test.shape}')


y_fit: (67994, 6)  y_val: (11999, 6)  y_test: (19999, 6)


## Step 2 — Compute scale_pos_weight per Label (Train Only)

In [4]:
# FIX F: per-label scale_pos_weight
spw = {}
print('scale_pos_weight per label (from fit set):')
for i, lbl in enumerate(LABELS):
    pos = y_fit[:, i].sum()
    neg = len(y_fit) - pos
    spw[lbl] = round(neg / max(pos, 1), 2)
    print(f'  {lbl:<20} pos={pos:,}  neg={neg:,}  spw={spw[lbl]}')

scale_pos_weight per label (from fit set):
  Anaemia              pos=10,531  neg=57,463  spw=5.46
  Diabetes_Risk        pos=5,532  neg=62,462  spw=11.29
  Dyslipidemia         pos=14,078  neg=53,916  spw=3.83
  Kidney_Risk          pos=13,101  neg=54,893  spw=4.19
  Liver_Stress         pos=18,242  neg=49,752  spw=2.73
  Thyroid_Abnormal     pos=16,749  neg=51,245  spw=3.06


## Step 3 — Train Ensemble Models (Per-Label Binary Relevance)

Each model is stored as `trained_models[model_name][label] = fitted_estimator`.  
Fitted on the fit subset only (85% of train).

In [5]:
# FIX F: per-label binary estimator dicts
trained_models = {'XGB_weighted': {}, 'RF_tuned': {}, 'LGBM_weighted': {}, 'AdaBoost': {}}

print('Training XGB_weighted (per label)...')
for i, lbl in enumerate(LABELS):
    cols = feature_cols_for_label(train_fit, lbl)
    X_tr = train_fit[cols].values
    est = XGBClassifier(
        n_estimators=300, max_depth=6,
        scale_pos_weight=spw[lbl],
        eval_metric='logloss',
        use_label_encoder=False,
        random_state=42, n_jobs=-1, verbosity=0
    )
    est.fit(X_tr, y_fit[:, i])
    trained_models['XGB_weighted'][lbl] = est

print('Training RF_tuned (per label)...')
for i, lbl in enumerate(LABELS):
    cols = feature_cols_for_label(train_fit, lbl)
    X_tr = train_fit[cols].values
    est = RandomForestClassifier(
        n_estimators=300, class_weight='balanced',
        max_depth=15, n_jobs=-1, random_state=42
    )
    est.fit(X_tr, y_fit[:, i])
    trained_models['RF_tuned'][lbl] = est

print('Training LGBM_weighted (per label)...')
for i, lbl in enumerate(LABELS):
    cols = feature_cols_for_label(train_fit, lbl)
    X_tr = train_fit[cols].values
    est = LGBMClassifier(
        n_estimators=300, max_depth=6,
        is_unbalance=True,
        random_state=42, n_jobs=-1, verbose=-1
    )
    est.fit(X_tr, y_fit[:, i])
    trained_models['LGBM_weighted'][lbl] = est

print('Training AdaBoost (per label)...')
for i, lbl in enumerate(LABELS):
    cols = feature_cols_for_label(train_fit, lbl)
    X_tr = train_fit[cols].values
    est = AdaBoostClassifier(n_estimators=200, random_state=42)
    est.fit(X_tr, y_fit[:, i])
    trained_models['AdaBoost'][lbl] = est

joblib.dump(trained_models['XGB_weighted'], f'{MODELS}/xgb_weighted.pkl')
joblib.dump(trained_models['RF_tuned'],     f'{MODELS}/rf_tuned.pkl')
joblib.dump(trained_models['LGBM_weighted'],f'{MODELS}/lgbm_weighted.pkl')
joblib.dump(trained_models['AdaBoost'],     f'{MODELS}/adaboost.pkl')
print('Primary models trained and saved.')


Training XGB_weighted (per label)...


Training RF_tuned (per label)...


Training LGBM_weighted (per label)...


Training AdaBoost (per label)...


Primary models trained and saved.


## Step 4 — XGB_SMOTE (Binary Relevance per Label)

SMOTE is applied **per label** on train data only, then XGBClassifier is trained without class weights.  
Included for paper comparison — expected to underperform class-weighted XGB.

In [6]:
# FIX F: XGB_SMOTE as per-label dict
trained_models['XGB_SMOTE'] = {}

for i, lbl in enumerate(LABELS):
    cols = feature_cols_for_label(train_fit, lbl)
    X_tr = train_fit[cols].values
    y_lbl = y_fit[:, i]
    pos_count = int(y_lbl.sum())
    k_neighbors = min(5, max(1, pos_count - 1))
    try:
        sm = SMOTE(random_state=42, k_neighbors=k_neighbors)
        X_sm, y_sm = sm.fit_resample(X_tr, y_lbl)
    except Exception as e:
        print(f'  SMOTE failed for {lbl} ({e}), using original data')
        X_sm, y_sm = X_tr, y_lbl
    xgb_s = XGBClassifier(
        n_estimators=300, max_depth=6,
        eval_metric='logloss', use_label_encoder=False,
        random_state=42, n_jobs=-1, verbosity=0
    )
    xgb_s.fit(X_sm, y_sm)
    trained_models['XGB_SMOTE'][lbl] = xgb_s
    print(f'  {lbl}: SMOTE train size {len(X_sm):,}')

joblib.dump(trained_models['XGB_SMOTE'], f'{MODELS}/xgb_smote.pkl')
print('XGB_SMOTE saved.')


  Anaemia: SMOTE train size 114,926


  Diabetes_Risk: SMOTE train size 124,924


  Dyslipidemia: SMOTE train size 107,832


  Kidney_Risk: SMOTE train size 109,786


  Liver_Stress: SMOTE train size 99,504


  Thyroid_Abnormal: SMOTE train size 102,490
XGB_SMOTE saved.


## Step 5 — Per-Label Threshold Tuning (Validation Set)

Thresholds are tuned on the **validation** set (15% of train).  
Never tuned on the fit set or the test set.

In [7]:
# FIX D: tune on validation set only
THRESHOLDS = np.arange(0.1, 0.95, 0.05)

def tune_per_label(model_name, model_dict, val_df, y_val_arr, labels, use_all_features=False):
    thresholds = {}
    tuning_records = []
    for i, lbl in enumerate(labels):
        est  = model_dict[lbl]
        cols = all_feature_cols(val_df) if use_all_features else feature_cols_for_label(val_df, lbl)
        X_val_lbl = val_df[cols].values
        try:
            p = est.predict_proba(X_val_lbl)[:, 1]
        except AttributeError:
            thresholds[lbl] = 0.5
            continue
        best_f1, best_t = 0.0, 0.5
        for t in THRESHOLDS:
            f1 = f1_score(y_val_arr[:, i], (p >= t).astype(int), zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        thresholds[lbl] = round(best_t, 2)
        tuning_records.append({
            'Model': model_name,
            'label': lbl,
            'Threshold': round(best_t, 2),
            'Val_F1': best_f1
        })
    return thresholds, tuning_records

thresholds = {}
all_tuning_records = []
for name, model_dict in trained_models.items():
    use_all = False
    model_thresholds, records = tune_per_label(name, model_dict, train_val, y_val, LABELS, use_all_features=use_all)
    thresholds[name] = model_thresholds
    if name in ['XGB_weighted', 'RF_tuned', 'LGBM_weighted']:
        all_tuning_records.extend(records)

pd.DataFrame(all_tuning_records).to_csv(f'{METRICS}/05_threshold_tuning.csv', index=False)

thr_df = pd.DataFrame(thresholds, index=LABELS)
print('Optimal thresholds (label x model, tuned on val):')
print(thr_df.to_string())


Optimal thresholds (label x model, tuned on val):
                  XGB_weighted  RF_tuned  LGBM_weighted  AdaBoost  XGB_SMOTE
Anaemia                   0.50      0.45           0.55      0.40       0.25
Diabetes_Risk             0.45      0.40           0.65      0.45       0.20
Dyslipidemia              0.35      0.40           0.45      0.35       0.20
Kidney_Risk               0.40      0.40           0.50      0.40       0.25
Liver_Stress              0.40      0.45           0.45      0.40       0.30
Thyroid_Abnormal          0.35      0.45           0.45      0.35       0.25


## Step 6 — Evaluation on Test Set

In [8]:
# FIX F: per-label evaluation
def eval_per_label_binary(model_name, model_dict, test_df, y_te_arr, thr_dict, labels):
    use_all = False
    rows, preds_all, proba_store = [], np.zeros((len(test_df), len(labels)), dtype=int), {}
    for i, lbl in enumerate(labels):
        est  = model_dict[lbl]
        cols = all_feature_cols(test_df) if use_all else feature_cols_for_label(test_df, lbl)
        X_te = test_df[cols].values
        try:
            p = est.predict_proba(X_te)[:, 1]
        except AttributeError:
            p = est.predict(X_te).astype(float)
        pred = (p >= thr_dict[lbl]).astype(int)
        preds_all[:, i] = pred
        proba_store[lbl] = p
        rows.append({
            'label':     lbl,
            'F1':        f1_score(y_te_arr[:, i], pred, zero_division=0),
            'Precision': precision_score(y_te_arr[:, i], pred, zero_division=0),
            'Recall':    recall_score(y_te_arr[:, i], pred, zero_division=0),
            'AUC_ROC':   roc_auc_score(y_te_arr[:, i], p),
            'PR_AUC':    average_precision_score(y_te_arr[:, i], p),
        })
    return pd.DataFrame(rows), preds_all, proba_store

model_results = {}
model_probas  = {}
for name, model_dict in trained_models.items():
    df_res, preds, probas = eval_per_label_binary(
        name, model_dict, test_scaled, y_test, thresholds[name], LABELS
    )
    model_results[name] = (df_res, preds)
    model_probas[name]  = probas

all_per_label = pd.concat(
    [df.assign(Model=name) for name, (df, _) in model_results.items()]
)[['Model','label','F1','Precision','Recall','AUC_ROC','PR_AUC']]
print(all_per_label.round(4).to_string(index=False))


        Model            label     F1  Precision  Recall  AUC_ROC  PR_AUC
 XGB_weighted          Anaemia 0.4202     0.4094  0.4316   0.7408  0.4347
 XGB_weighted    Diabetes_Risk 0.4092     0.3905  0.4298   0.8056  0.4118
 XGB_weighted     Dyslipidemia 0.3999     0.3009  0.5958   0.6725  0.4049
 XGB_weighted      Kidney_Risk 0.4257     0.3505  0.5418   0.7077  0.4293
 XGB_weighted     Liver_Stress 0.5353     0.4539  0.6521   0.7511  0.5604
 XGB_weighted Thyroid_Abnormal 0.4722     0.3657  0.6663   0.7108  0.4801
     RF_tuned          Anaemia 0.4033     0.3886  0.4193   0.7368  0.4093
     RF_tuned    Diabetes_Risk 0.3925     0.3498  0.4471   0.8114  0.3903
     RF_tuned     Dyslipidemia 0.4100     0.3009  0.6433   0.6850  0.4031
     RF_tuned      Kidney_Risk 0.4319     0.3350  0.6078   0.7300  0.4536
     RF_tuned     Liver_Stress 0.5444     0.4621  0.6625   0.7703  0.5800
     RF_tuned Thyroid_Abnormal 0.4957     0.3898  0.6808   0.7395  0.5163
LGBM_weighted          Anaemia 0.4229 

In [9]:
# Overall multi-label metrics table
overall_rows = []
for name, (df, preds) in model_results.items():
    overall_rows.append({
        'Model':           name,
        'Hamming_Loss':    hamming_loss(y_test, preds),
        'Subset_Accuracy': accuracy_score(y_test, preds),
        'Micro_F1':        f1_score(y_test, preds, average='micro', zero_division=0),
        'Macro_F1':        f1_score(y_test, preds, average='macro', zero_division=0),
    })
overall_df = pd.DataFrame(overall_rows).set_index('Model')
print('=== Overall Metrics ===')
print(overall_df.round(4).to_string())

=== Overall Metrics ===
               Hamming_Loss  Subset_Accuracy  Micro_F1  Macro_F1
Model                                                           
XGB_weighted         0.2656           0.1928    0.4544    0.4437
RF_tuned             0.2700           0.1895    0.4607    0.4463
LGBM_weighted        0.2804           0.1627    0.4607    0.4496
AdaBoost             0.4336           0.0272    0.4019    0.3929
XGB_SMOTE            0.2560           0.2154    0.4559    0.4471


## Step 7 — Visualisations

In [10]:
# PR curves — per-label probabilities from per-label estimators
COLORS = {'XGB_weighted':'#e74c3c','RF_tuned':'#2ecc71','LGBM_weighted':'#f39c12','AdaBoost':'#3498db','XGB_SMOTE':'#9b59b6'}

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for i, lbl in enumerate(LABELS):
    ax = axes[i // 3][i % 3]
    for name, probas in model_probas.items():
        p = probas[lbl]
        prec, rec, _ = precision_recall_curve(y_test[:, i], p)
        auc = average_precision_score(y_test[:, i], p)
        ls = '--' if name == 'XGB_SMOTE' else '-'
        ax.plot(rec, prec, label=f'{name} (AP={auc:.2f})', color=COLORS[name], lw=1.5, linestyle=ls)
    ax.set_title(lbl, fontsize=11)
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.legend(fontsize=7)
plt.suptitle('Precision-Recall Curves — All Ensemble Models', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f'{PLOTS}/05_pr_curves.png', bbox_inches='tight')
plt.show()

In [11]:
# Grouped bar chart: Macro F1 per model
macro_f1 = {name: f1_score(y_test, preds, average='macro', zero_division=0)
             for name, (_, preds) in model_results.items()}

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(macro_f1.keys(), macro_f1.values(),
              color=[COLORS[m] for m in macro_f1], edgecolor='white', width=0.5)
for bar, val in zip(bars, macro_f1.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Macro F1 Score by Model (Test Set)', fontsize=13)
ax.set_ylabel('Macro F1')
ax.set_ylim(0, max(macro_f1.values()) * 1.15)
plt.tight_layout()
plt.savefig(f'{PLOTS}/05_macro_f1_bar.png', bbox_inches='tight')
plt.show()

In [12]:
# F1 heatmap — label x model
f1_pivot = all_per_label.pivot(index='label', columns='Model', values='F1')
fig, ax = plt.subplots(figsize=(11, 6))
sns.heatmap(f1_pivot, annot=True, fmt='.3f', cmap='YlGn',
            linewidths=0.5, ax=ax, vmin=0, vmax=1)
ax.set_title('F1 Score — Label × Model', fontsize=13)
plt.tight_layout()
plt.savefig(f'{PLOTS}/05_f1_heatmap.png', bbox_inches='tight')
plt.show()

## SMOTE Analysis

Comparing **XGB_SMOTE** against **XGB_weighted** (best primary model) across all labels.

In [13]:
res_xgb,   _ = model_results['XGB_weighted']
res_smote, _ = model_results['XGB_SMOTE']

smote_compare = pd.DataFrame({
    'Label':              LABELS,
    'XGB_weighted_F1':    res_xgb['F1'].values,
    'XGB_SMOTE_F1':       res_smote['F1'].values,
    'XGB_weighted_Recall':res_xgb['Recall'].values,
    'XGB_SMOTE_Recall':   res_smote['Recall'].values,
})
smote_compare['F1_delta']     = (smote_compare['XGB_SMOTE_F1']
                                 - smote_compare['XGB_weighted_F1'])
smote_compare['Recall_delta'] = (smote_compare['XGB_SMOTE_Recall']
                                 - smote_compare['XGB_weighted_Recall'])
print('SMOTE vs XGB_weighted comparison:')
print(smote_compare.round(4).to_string(index=False))

SMOTE vs XGB_weighted comparison:
           Label  XGB_weighted_F1  XGB_SMOTE_F1  XGB_weighted_Recall  XGB_SMOTE_Recall  F1_delta  Recall_delta
         Anaemia           0.4202        0.4246               0.4316            0.4736    0.0044        0.0420
   Diabetes_Risk           0.4092        0.4173               0.4298            0.4764    0.0081        0.0466
    Dyslipidemia           0.3999        0.4069               0.5958            0.5967    0.0070        0.0010
     Kidney_Risk           0.4257        0.4234               0.5418            0.5009   -0.0023       -0.0409
    Liver_Stress           0.5353        0.5342               0.6521            0.6195   -0.0011       -0.0327
Thyroid_Abnormal           0.4722        0.4762               0.6663            0.6102    0.0040       -0.0561


### Interpretation

> *"SMOTE underperformed class-weighted XGBoost across all labels, likely because synthetic oversampling*
> *in high-dimensional clinical lab space generates unrealistic patient profiles. When SMOTE creates*
> *synthetic minority samples by interpolating between real patients in a 60+ dimensional feature space,*
> *the resulting vectors do not correspond to physiologically plausible lab result combinations.*
> *XGBoost's native `scale_pos_weight` — which up-weights real minority examples during loss computation —*
> *achieves better calibration without introducing synthetic artefacts.*
> *SMOTE underperformance is reported as a finding in this project."*

## Step 8 — Save Metrics & Best Model Summary

In [14]:
all_per_label.to_csv(f'{METRICS}/05_ensemble_results.csv', index=False)
overall_df.reset_index().to_csv(f'{METRICS}/05_overall_metrics.csv', index=False)
print('Results saved.')

# Best model per label
best_per_label = all_per_label.loc[
    all_per_label.groupby('label')['F1'].idxmax()
][['label','Model','F1','Precision','Recall','AUC_ROC']]
best_per_label.to_csv(f'{METRICS}/05_best_models.csv', index=False)

print('\n=== Best Model per Label ===')
print(best_per_label.round(4).to_string(index=False))
print('\n=== Overall — Recommended Model (XGB_weighted) ===')
print(overall_df.loc[['XGB_weighted']].round(4).to_string())
print('\nThe paper cites XGB_weighted as the recommended model.')
print('SMOTE underperformance is cited as a limitation/finding.')

Results saved.

=== Best Model per Label ===
           label         Model     F1  Precision  Recall  AUC_ROC
         Anaemia  XGB_weighted 0.4202     0.4094  0.4316   0.7408
         Anaemia      RF_tuned 0.4033     0.3886  0.4193   0.7368
         Anaemia LGBM_weighted 0.4229     0.3848  0.4693   0.7556
         Anaemia      AdaBoost 0.3356     0.2211  0.6958   0.6884
         Anaemia     XGB_SMOTE 0.4246     0.3848  0.4736   0.7469
   Diabetes_Risk  XGB_weighted 0.4092     0.3905  0.4298   0.8056
   Diabetes_Risk      RF_tuned 0.3925     0.3498  0.4471   0.8114
   Diabetes_Risk LGBM_weighted 0.4220     0.3923  0.4566   0.8284
   Diabetes_Risk      AdaBoost 0.3635     0.3521  0.3756   0.7965
   Diabetes_Risk     XGB_SMOTE 0.4173     0.3713  0.4764   0.8185
    Dyslipidemia  XGB_weighted 0.3999     0.3009  0.5958   0.6725
    Dyslipidemia      RF_tuned 0.4100     0.3009  0.6433   0.6850
    Dyslipidemia LGBM_weighted 0.4095     0.3010  0.6404   0.6816
    Dyslipidemia      AdaBoost 